# Tesla and GameStop Stock & Revenue Dashboard
**Project:** Stock Price vs Revenue Analysis using yfinance and Web Scraping  
**Course:** Peer-Graded Final Project

This notebook collects historical stock prices for Tesla (TSLA) and GameStop (GME), extracts revenue tables from the web, displays the requested rows, and plots stock price dashboards.

> Run the cells from top to bottom. If a website blocks automated scraping, the scraping cell includes an alternate public data source. Take screenshots of each question heading together with its code and output for your submission.

## Setup

In [ ]:
# If needed, uncomment and run this installation cell:
# %pip install yfinance pandas requests beautifulsoup4 lxml html5lib plotly

## Question 1: Use yfinance to Extract Tesla Stock Data

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go

# Extract Tesla stock data
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")

# Reset the index and display the first five rows
tesla_data.reset_index(inplace=True)
tesla_data.head()

## Question 2: Use Webscraping to Extract Tesla Revenue Data

In [ ]:
def extract_revenue(symbol, company_name):
    \"\"\"Scrape quarterly revenue data from Macrotrends, with a Stock Analysis fallback.\"\"\"
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"
    }
    urls = [
        f"https://www.macrotrends.net/stocks/charts/{symbol}/{company_name}/revenue",
        f"https://stockanalysis.com/stocks/{symbol.lower()}/revenue/"
    ]

    last_error = None
    for url in urls:
        try:
            response = requests.get(url, headers=headers, timeout=30)
            response.raise_for_status()
            tables = pd.read_html(response.text)
            for table in tables:
                # Identify a table containing a date/period column and a revenue column
                cols = [str(c).strip().lower() for c in table.columns]
                date_col = next((c for c in table.columns if any(k in str(c).lower() for k in ["date", "period", "quarter"])), None)
                revenue_col = next((c for c in table.columns if "revenue" in str(c).lower()), None)
                if date_col is not None and revenue_col is not None:
                    result = table[[date_col, revenue_col]].copy()
                    result.columns = ["Date", "Revenue"]
                    result = result.dropna()
                    result["Revenue"] = (result["Revenue"].astype(str)
                                         .str.replace(r"[$,]", "", regex=True)
                                         .str.replace(r"\s+", "", regex=True))
                    result = result[result["Revenue"].str.match(r"^-?\d+(\.\d+)?$", na=False)]
                    result["Revenue"] = result["Revenue"].astype(float)
                    return result.reset_index(drop=True)
        except Exception as e:
            last_error = e
    raise RuntimeError(
        f"Could not retrieve {symbol} revenue data from the available websites. "
        f"Check your internet connection or try again later. Last error: {last_error}"
    )

# Extract Tesla revenue data
tesla_revenue = extract_revenue("TSLA", "tesla")
tesla_revenue.tail()

## Question 3: Use yfinance to Extract GameStop Stock Data

In [ ]:
# Extract GameStop stock data
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")

# Reset the index and display the first five rows
gme_data.reset_index(inplace=True)
gme_data.head()

## Question 4: Use Webscraping to Extract GME Revenue Data

In [ ]:
# Extract GameStop revenue data
gme_revenue = extract_revenue("GME", "gamestop")
gme_revenue.tail()

## Question 5: Plot Tesla Stock Graph

In [ ]:
from plotly.subplots import make_subplots

def make_graph(stock_data, revenue_data, stock_name):
    # Prepare stock data
    stock = stock_data.copy()
    if "Date" not in stock.columns:
        stock = stock.reset_index()
    stock["Date"] = pd.to_datetime(stock["Date"])

    # Prepare revenue data
    revenue = revenue_data.copy()
    revenue["Date"] = pd.to_datetime(revenue["Date"], errors="coerce")
    revenue = revenue.dropna(subset=["Date"])
    revenue["Revenue"] = pd.to_numeric(revenue["Revenue"], errors="coerce")
    revenue = revenue.dropna(subset=["Revenue"])

    # Create a two-panel dashboard: share price and revenue
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        subplot_titles=(f"{stock_name} Historical Share Price",
                        f"{stock_name} Quarterly Revenue"),
        vertical_spacing=0.15
    )
    fig.add_trace(
        go.Scatter(x=stock["Date"], y=stock["Close"],
                   mode="lines", name="Share Price"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=revenue["Date"], y=revenue["Revenue"],
                   mode="lines+markers", name="Revenue"),
        row=2, col=1
    )
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Share Price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD)", row=2, col=1)
    fig.update_layout(
        title_text=f"{stock_name} Stock Price vs Revenue Dashboard",
        height=750,
        template="plotly_white",
        hovermode="x unified"
    )
    fig.show()

make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6: Plot GameStop Stock Graph

In [ ]:
make_graph(gme_data, gme_revenue, "GameStop")

## Question 7: Share your Assignment Notebook

1. Save this notebook as `Tesla_GME_Stock_Revenue_Dashboard.ipynb`.
2. Upload the `.ipynb` file to a GitHub repository.
3. Open the uploaded notebook on GitHub and copy its browser URL.
4. Paste that URL into the assignment's **URL** field.

**Suggested GitHub repository name:** `Tesla-GME-Stock-Revenue-Dashboard`

**Suggested repository description:** `Python final project comparing Tesla and GameStop historical stock prices and scraped revenue data using yfinance, pandas, BeautifulSoup, and Plotly.`

### Submission screenshot checklist
- Q1: Tesla code and `tesla_data.head()` output.
- Q2: `tesla_revenue.tail()` output.
- Q3: GameStop code and `gme_data.head()` output.
- Q4: `gme_revenue.tail()` output.
- Q5: Tesla stock graph.
- Q6: GameStop stock graph.
- Q7: GitHub URL of the uploaded notebook.

**Important:** Run every cell successfully and capture your own executed outputs. Do not upload screenshots of code without its output.